In [9]:
# Import dependencies

import kagglehub
import os
import pandas as pd

In [10]:
# Download the latest dataset version
path = kagglehub.dataset_download("harshadapatil31/student-performance-and-study-habits-dataset")

print("Path to dataset files:", path)

files = os.listdir(path)

csv_files = []
for file in files:
    if file.lower().endswith('.csv'):
        csv_files.append(file)

if not csv_files:
    raise FileNotFoundError('No CSV file found in the downloaded dataset')

csv_path = os.path.join(path, csv_files[0])

df = pd.read_csv(csv_path)

Path to dataset files: C:\Users\Administrador\.cache\kagglehub\datasets\harshadapatil31\student-performance-and-study-habits-dataset\versions\1


In [11]:
print(f'Head:\n{df.head()}\n')

print(f'Describe:\n{df.describe()}\n')

Head:
   student_id  gender  study_time_hours  attendance_percent  sleep_hours  \
0           1    Male               4.0                98.0          6.5   
1           2  Female               6.3               100.0          5.7   
2           3    Male               4.9                85.3          7.9   
3           4    Male               2.6                77.5          8.0   
4           5    Male               2.2                89.6          4.6   

  parental_education internet_access extracurricular_activities part_time_job  \
0          Bachelors             Yes                        Yes            No   
1        High School             Yes                        Yes           Yes   
2          Bachelors             Yes                         No           Yes   
3                NaN             Yes                        Yes            No   
4          Bachelors             Yes                         No           Yes   

   previous_grade  final_exam_score final_grade  


In [12]:

print(f'dfisnull: {df.isnull().sum()}\n')

print(df.dtypes)

print(f'dfshape: {df.shape}\n')

print(f'dfcolumns: {df.columns}\n')

print(f'duplicated rows: {df.duplicated().sum()}\n')

dfisnull: student_id                      0
gender                          0
study_time_hours                0
attendance_percent              0
sleep_hours                     0
parental_education            102
internet_access                 0
extracurricular_activities      0
part_time_job                   0
previous_grade                  0
final_exam_score                0
final_grade                     0
dtype: int64

student_id                      int64
gender                            str
study_time_hours              float64
attendance_percent            float64
sleep_hours                   float64
parental_education                str
internet_access                   str
extracurricular_activities        str
part_time_job                     str
previous_grade                float64
final_exam_score              float64
final_grade                       str
dtype: object
dfshape: (1000, 12)

dfcolumns: Index(['student_id', 'gender', 'study_time_hours', 'attendance_per

The initial data-quality checks found missing values in `parental_education` and no duplicated rows.

## Handling missing values

There are three possible approaches to the missing values in `parental_education`:

1. Drop the entire column.
2. Replace missing values with the column mode.
3. Replace missing values with a new category.

There are 102 missing values, approximately 10% of the dataset. Because the missingness itself may carry information, the Silver layer preserves those records and uses `Unknown` as a separate category.

In [13]:
df['parental_education'] = df['parental_education'].fillna('Unknown')

In [14]:
categorical_columns = [
    'gender',
    'parental_education',
    'internet_access',
    'extracurricular_activities',
    'part_time_job',
    'final_grade'
]

for column in categorical_columns:
    df[column] = df[column].str.strip().str.title()
    print(column, df[column].unique(), '\n')

# final_grade is an ordinal categorical variable

gender <ArrowStringArray>
['Male', 'Female']
Length: 2, dtype: str 

parental_education <ArrowStringArray>
['Bachelors', 'High School', 'Unknown', 'Masters', 'Phd']
Length: 5, dtype: str 

internet_access <ArrowStringArray>
['Yes', 'No']
Length: 2, dtype: str 

extracurricular_activities <ArrowStringArray>
['Yes', 'No']
Length: 2, dtype: str 

part_time_job <ArrowStringArray>
['No', 'Yes']
Length: 2, dtype: str 

final_grade <ArrowStringArray>
['A', 'B', 'D', 'C', 'F']
Length: 5, dtype: str 



The following assertions validate the cleaned Silver dataset against its expected schema and business rules.

In [15]:
df = df.drop_duplicates()

assert df.duplicated().sum() == 0
assert df['student_id'].notna().all()
assert df['student_id'].is_unique
assert df['attendance_percent'].between(0, 100).all()
assert df['study_time_hours'].ge(0).all()
assert df['sleep_hours'].ge(0).all()
assert df['previous_grade'].between(0, 100).all()
assert df['final_exam_score'].between(0, 100).all()
assert df['parental_education'].notna().all()
assert df.isna().sum().sum() == 0

assert set(df['internet_access'].unique()) <= {'Yes', 'No'}
assert set(df['extracurricular_activities'].unique()) <= {'Yes', 'No'}
assert set(df['part_time_job'].unique()) <= {'Yes', 'No'}
assert set(df['final_grade'].unique()) <= {'A', 'B', 'C', 'D', 'F'}

In [16]:
os.makedirs('./data/silver', exist_ok=True)

df.to_parquet('./data/silver/student_habits_silver.parquet', index=False)

print('Silver layer saved successfully')

Silver layer saved successfully
